# modern-yolonas — Quickstart

A clean, minimal reimplementation of [YOLO-NAS](https://github.com/Deci-AI/super-gradients)
object detection. No factory patterns, no registries, no OmegaConf — just PyTorch.

This notebook covers, end to end:

1. Installing the package
2. Detecting objects in an image with the high-level `Detector` API
3. Visualizing the result inline
4. Dropping down to the raw model to see what the network actually outputs
5. Exporting to ONNX / OpenVINO for deployment

Runs on CPU. A GPU makes it faster but is not required.

---

## ⚠️ Read this before you use the pretrained weights

**The code in this library is MIT. The pretrained COCO weights are not.**

The COCO checkpoints are converted from Deci AI's super-gradients releases and remain under
the [Super Gradients Model EULA](https://github.com/Deci-AI/super-gradients/blob/master/LICENSE.YOLONAS.md),
which is **non-commercial use only**. This trips people up constantly, because the
super-gradients *code* is Apache-2.0 and it is easy to assume the weights inherit that.
They do not.

For a commercial deployment, train from scratch on your own data (see `yolonas train`).
This notebook uses the pretrained weights for demonstration only.

## 1. Install

Requires **Python 3.10+**. Uncomment whichever line matches your setup.

In [ ]:
# %pip install modern-yolonas matplotlib

# Optional extras:
#   %pip install "modern-yolonas[onnx]"      # ONNX export
#   %pip install "modern-yolonas[openvino]"  # OpenVINO export
#
# With uv (faster):
#   !uv pip install modern-yolonas matplotlib

In [ ]:
import torch
import modern_yolonas

print(f"modern-yolonas {modern_yolonas.__version__}")
print(f"torch          {torch.__version__}")
print(f"CUDA available {torch.cuda.is_available()}")

## 2. Grab a test image

A COCO `val2017` frame — two cats asleep on a couch, with two TV remotes next to them.
It is a good smoke test because it exercises three things at once: multiple classes, two
instances of the *same* class (so NMS has to keep both), and one small object (the
remotes) alongside one very large one (the couch).

In [ ]:
import urllib.request
from pathlib import Path

# COCO val2017, image id 000000039769
IMAGE_URL = "http://images.cocodataset.org/val2017/000000039769.jpg"
image_path = Path("cats.jpg")

if not image_path.exists():
    urllib.request.urlretrieve(IMAGE_URL, image_path)

print(f"{image_path} — {image_path.stat().st_size / 1024:.0f} KB")

## 3. Detect

`Detector` wraps the whole pipeline — load → preprocess → forward → NMS → rescale boxes
back to original image coordinates.

The first call downloads the checkpoint from the Hugging Face Hub and caches it, so it is
slow once and fast forever after. **You will see a licence warning** — that is the EULA
note from the top of this notebook, printed deliberately.

In [ ]:
from modern_yolonas import Detector

device = "cuda" if torch.cuda.is_available() else "cpu"

det = Detector("yolo_nas_s", device=device, conf_threshold=0.25)
result = det(image_path)

print(f"{len(result.boxes)} objects detected")

In [ ]:
from modern_yolonas.inference.visualize import COCO_NAMES

for box, score, class_id in zip(result.boxes, result.scores, result.class_ids):
    x1, y1, x2, y2 = box
    name = COCO_NAMES[int(class_id)]
    print(f"{name:>10}  {score:.2f}   [{x1:5.0f}, {y1:5.0f}, {x2:5.0f}, {y2:5.0f}]")

### What came back

`result` is a `Detection` dataclass holding three parallel numpy arrays:

| Attribute | Shape | Meaning |
|---|---|---|
| `result.boxes` | `[D, 4]` | `x1, y1, x2, y2` in **original image pixels** |
| `result.scores` | `[D]` | confidence in `[0, 1]` |
| `result.class_ids` | `[D]` | COCO-80 class index |

`D` is however many boxes survived NMS — it is not a fixed size, and it changes with
`conf_threshold`.

In [ ]:
print(f"boxes     {result.boxes.shape}  {result.boxes.dtype}")
print(f"scores    {result.scores.shape}  {result.scores.dtype}")
print(f"class_ids {result.class_ids.shape}  {result.class_ids.dtype}")

## 4. Visualize

`result.visualize()` returns an annotated **BGR** numpy array (OpenCV convention), so flip
the channel order before handing it to matplotlib.

In [ ]:
import matplotlib.pyplot as plt

annotated_bgr = result.visualize()

plt.figure(figsize=(10, 7))
plt.imshow(annotated_bgr[:, :, ::-1])  # BGR -> RGB
plt.axis("off")
plt.tight_layout()
plt.show()

# Or straight to disk:
# result.save("output.jpg")

### The threshold is a dial, not a setting

`conf_threshold` decides how much marginal evidence you are willing to accept. Drop it and
you buy recall with precision — extra boxes appear, and the low-scoring ones are usually
wrong. Worth running once so you can see the trade-off rather than take it on faith.

In [ ]:
for conf in (0.50, 0.25, 0.10, 0.05):
    r = det(image_path, conf_threshold=conf)
    names = [COCO_NAMES[int(c)] for c in r.class_ids]
    print(f"conf={conf:.2f} -> {len(r.boxes):2d} boxes  {names}")

## 5. Under the hood

The `Detector` is a convenience wrapper. The model itself is an ordinary `nn.Module` you
can call directly — useful for batching, custom postprocessing, or wiring YOLO-NAS into a
larger pipeline.

YOLO-NAS is **anchor-free**: it predicts one box per feature-map cell, across three
detection levels at strides 8, 16 and 32. At 640×640 that is

$$\left(\tfrac{640}{8}\right)^2 + \left(\tfrac{640}{16}\right)^2 + \left(\tfrac{640}{32}\right)^2
= 80^2 + 40^2 + 20^2 = 6400 + 1600 + 400 = 8400$$

predictions — which is exactly the middle dimension you will see below. The `assert` at the
end is there to make the point rather than to guard anything.

In [ ]:
from modern_yolonas import yolo_nas_s

model = yolo_nas_s(pretrained=True).eval().to(device)

x = torch.randn(1, 3, 640, 640, device=device)
with torch.no_grad():
    pred_boxes, pred_scores = model(x)

print(f"pred_boxes  {tuple(pred_boxes.shape)}   x1y1x2y2, pixel coords")
print(f"pred_scores {tuple(pred_scores.shape)}   per-class probabilities")

n_params = sum(p.numel() for p in model.parameters())
print(f"\nparameters  {n_params / 1e6:.2f}M")

assert pred_boxes.shape[1] == 80**2 + 40**2 + 20**2 == 8400

Note there is **no objectness score** and **no NMS** at this level — the raw head emits
8400 candidate boxes with 80 class probabilities each, and it is `postprocess()` that
filters them down to the handful you saw in step 3.

## 6. Video

Two options — write an annotated file directly, or iterate frames and do your own thing
with each result. (Not executed here; point it at a video you have.)

In [ ]:
# Option 1 — annotate straight to a file
# stats = det.detect_video_to_file("input.mp4", "output.mp4")
# print(f"{stats['total_detections']} detections over {stats['total_frames']} frames")

# Option 2 — iterate, for custom per-frame logic (tracking, counting, alerting)
# for frame_idx, frame_result in det.detect_video("input.mp4"):
#     print(f"frame {frame_idx}: {len(frame_result.boxes)} objects")

## 7. Export for deployment

ONNX and OpenVINO exports are available from both Python and the CLI. These need the
matching extra installed (`modern-yolonas[onnx]` / `modern-yolonas[openvino]`).

In [ ]:
# !yolonas export --model yolo_nas_s --format onnx     --output yolo_nas_s.onnx
# !yolonas export --model yolo_nas_s --format openvino --output yolo_nas_s.xml

# Frigate NVR target — bakes preprocessing AND NMS into the graph, so the exported
# model takes raw uint8 BGR in and emits a flat [D, 7] tensor:
#   [batch, x1, y1, x2, y2, confidence, class_id]
# !yolonas export --model yolo_nas_s --format openvino --target frigate --input-size 320

## Where to go next

| Task | Command |
|---|---|
| Detect on a folder | `yolonas detect --model yolo_nas_s --source images/ --output results/` |
| Fine-tune on your data | `yolonas train --model yolo_nas_s --data ./dataset --format yolo --epochs 100` |
| Evaluate on COCO | `yolonas eval --model yolo_nas_s --data ./coco --split val2017` |
| Benchmark latency | `yolonas benchmark --model yolo_nas_s` |

Model variants. Parameter counts are measured with
`sum(p.numel() for p in model.parameters())`; the mAP column is Deci's **published**
figure, which this project has not independently re-measured.

| Model | Params (measured) | Input | mAP (published) |
|---|---|---|---|
| `yolo_nas_s` | 19.05M | 640 | 47.5 |
| `yolo_nas_m` | 51.18M | 640 | 51.5 |
| `yolo_nas_l` | 66.98M | 640 | 52.2 |

- Full docs: [`docs/`](../docs/index.md)
- Runnable scripts: [`examples/`](../examples/)
- Architecture walkthrough: [`docs/architecture.md`](../docs/architecture.md)

And once more, because it is the thing people miss: **the pretrained COCO weights are
non-commercial.** Train from scratch for anything you intend to ship.